### (Modelo corrido no Kaggle GPU T4 x2)

# Stricture vs Lithiasis specialist

This notebook tests a targeted specialist for the largest remaining confusion in the final model:

```text
Stricture → Lithiasis
```

The specialist is trained only to distinguish `Stricture` from `Lithiasis`. It is then used as a conservative override only when the base blend predicts either `Lithiasis` or `Stricture` and the specialist is sufficiently confident.


In [1]:
!pip -q install monai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 30.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 105.0 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompa

## 1. Setup: imports, constants and paths

In [ ]:
from pathlib import Path
import copy
import time
import random
from contextlib import nullcontext

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision.models import efficientnet_b7, efficientnet_b0

from monai.transforms import (
    Compose,
    LoadImage,
    EnsureChannelFirst,
    Resize,
    NormalizeIntensity,
    Lambda,
    ToTensor
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True

CLASS_NAMES = ["Biliary_Leaks", "Lithiasis", "Normal", "Stricture"]
NUM_CLASSES = len(CLASS_NAMES)

POSITIVE_CLASS = "Biliary_Leaks"
POSITIVE_IDX = CLASS_NAMES.index(POSITIVE_CLASS)

REAL_DATASET_ROOT = Path("/kaggle/input/datasets/martimflix/dataset_monica/dataset_monica")
GENERATED_ROOT = Path("/kaggle/input/datasets/martimflix/vae-generated/vae_generated_perceptual")

REAL_TRAIN_DIR = REAL_DATASET_ROOT / "train"
VAL_DIR = REAL_DATASET_ROOT / "val"
TEST_DIR = REAL_DATASET_ROOT / "test"

EFFICIENTNET_CKPT_PATH = Path(
    "/kaggle/input/models/martimflix/efficientnet/pytorch/default/1/efficientnet_b7.pth"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)


<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
2026-05-31 09:32:05.577811: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780219926.067544      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780219926.180714      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780219927.210417      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780219927.210457      58 computation_placer.cc:1

Device: cuda


## 2. Dataset verification

In [3]:
def count_images(folder):
    exts = ["*.png", "*.jpg", "*.jpeg", "*.bmp", "*.tif", "*.tiff"]
    files = []

    if not folder.exists():
        return 0

    for ext in exts:
        files.extend(folder.glob(ext))

    return len(files)


rows = []

for cls in CLASS_NAMES:
    rows.append({
        "Class": cls,
        "Real train": count_images(REAL_TRAIN_DIR / cls),
        "Generated": count_images(GENERATED_ROOT / cls),
        "Val real": count_images(VAL_DIR / cls),
        "Test real": count_images(TEST_DIR / cls)
    })

counts_df = pd.DataFrame(rows)
counts_df

,Class,Real train,Generated,Val real,Test real
0,Biliary_Leaks,110,300,24,17
1,Lithiasis,505,0,98,123
2,Normal,197,300,59,43
3,Stricture,255,300,53,84


## 3. Image transforms

In [4]:
def repeat_if_needed(img):
    if img.shape[0] == 1:
        return img.repeat(3, 1, 1)
    return img


train_transforms = Compose([
    LoadImage(image_only=True),
    EnsureChannelFirst(),
    Resize((512, 512)),
    NormalizeIntensity(),
    Lambda(repeat_if_needed),
    ToTensor()
])

eval_transforms = Compose([
    LoadImage(image_only=True),
    EnsureChannelFirst(),
    Resize((512, 512)),
    NormalizeIntensity(),
    Lambda(repeat_if_needed),
    ToTensor()
])

print("Transforms ready.")

Transforms ready.


## 4. General evaluation data loaders

In [5]:
USE_GENERATED_BILIARY = True


def collect_binary_paths(root_dir):
    image_files = []
    labels = []

    for cls in CLASS_NAMES:
        cls_dir = root_dir / cls

        cls_files = []
        for ext in ["*.png", "*.jpg", "*.jpeg", "*.bmp", "*.tif", "*.tiff"]:
            cls_files.extend(cls_dir.glob(ext))

        cls_files = sorted(cls_files)

        binary_label = 1 if cls == POSITIVE_CLASS else 0

        image_files.extend(cls_files)
        labels.extend([binary_label] * len(cls_files))

    return image_files, labels


def collect_binary_train_paths(real_train_dir, generated_root, use_generated_biliary=False):
    image_files, labels = collect_binary_paths(real_train_dir)

    if use_generated_biliary:
        gen_dir = generated_root / POSITIVE_CLASS

        generated_files = []
        for ext in ["*.png", "*.jpg", "*.jpeg", "*.bmp", "*.tif", "*.tiff"]:
            generated_files.extend(gen_dir.glob(ext))

        generated_files = sorted(generated_files)

        image_files.extend(generated_files)
        labels.extend([1] * len(generated_files))

    return image_files, labels


class BinaryPathDataset(Dataset):
    def __init__(self, image_files, labels, transform):
        self.image_files = list(image_files)
        self.labels = list(labels)
        self.transform = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img = self.transform(str(self.image_files[idx]))
        label = torch.tensor(float(self.labels[idx]), dtype=torch.float32)
        return img, label


binary_train_files, binary_train_labels = collect_binary_train_paths(
    real_train_dir=REAL_TRAIN_DIR,
    generated_root=GENERATED_ROOT,
    use_generated_biliary=USE_GENERATED_BILIARY
)

binary_val_files, binary_val_labels = collect_binary_paths(VAL_DIR)
binary_test_files, binary_test_labels = collect_binary_paths(TEST_DIR)

binary_train_ds = BinaryPathDataset(
    binary_train_files,
    binary_train_labels,
    train_transforms
)

binary_val_ds = BinaryPathDataset(
    binary_val_files,
    binary_val_labels,
    eval_transforms
)

binary_test_ds = BinaryPathDataset(
    binary_test_files,
    binary_test_labels,
    eval_transforms
)

binary_counts_df = pd.DataFrame([
    {
        "Split": "Train",
        "Biliary_Leaks": int(sum(binary_train_labels)),
        "Rest": int(len(binary_train_labels) - sum(binary_train_labels)),
        "Total": len(binary_train_labels)
    },
    {
        "Split": "Validation",
        "Biliary_Leaks": int(sum(binary_val_labels)),
        "Rest": int(len(binary_val_labels) - sum(binary_val_labels)),
        "Total": len(binary_val_labels)
    },
    {
        "Split": "Test",
        "Biliary_Leaks": int(sum(binary_test_labels)),
        "Rest": int(len(binary_test_labels) - sum(binary_test_labels)),
        "Total": len(binary_test_labels)
    }
])

binary_counts_df

,Split,Biliary_Leaks,Rest,Total
0,Train,410,957,1367
1,Validation,24,210,234
2,Test,17,250,267


In [6]:
BATCH_SIZE = 8

binary_train_loader = DataLoader(
    binary_train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

binary_val_loader = DataLoader(
    binary_val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

binary_test_loader = DataLoader(
    binary_test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

print("Binary loaders ready.")
print("Train batches:", len(binary_train_loader))
print("Val batches:", len(binary_val_loader))
print("Test batches:", len(binary_test_loader))

Binary loaders ready.
Train batches: 171
Val batches: 30
Test batches: 34


## 5. Checkpoint loading utilities

In [7]:
def clean_state_dict_keys(state_dict):
    cleaned = {}

    for k, v in state_dict.items():
        new_k = k

        if new_k.startswith("module."):
            new_k = new_k[len("module."):]

        cleaned[new_k] = v

    return cleaned


def safe_torch_load(path):
    try:
        return torch.load(
            path,
            map_location="cpu",
            weights_only=False
        )
    except TypeError:
        return torch.load(
            path,
            map_location="cpu"
        )


def load_checkpoint_safely(model, checkpoint_path):
    checkpoint = safe_torch_load(checkpoint_path)

    if isinstance(checkpoint, dict):
        if "state_dict" in checkpoint:
            state_dict = checkpoint["state_dict"]
        elif "model_state_dict" in checkpoint:
            state_dict = checkpoint["model_state_dict"]
        else:
            state_dict = checkpoint
    else:
        state_dict = checkpoint

    state_dict = clean_state_dict_keys(state_dict)

    missing, unexpected = model.load_state_dict(state_dict, strict=False)

    print("Checkpoint loaded.")
    print("Missing keys:", len(missing))
    print("Unexpected keys:", len(unexpected))

    return model

## 6. Multiclass helpers and labels

In [8]:

from torchvision.models import mobilenet_v2, resnet50
from monai.networks.nets import DenseNet121 as MonaiDenseNet121


def collect_multiclass_paths(root_dir):
    image_files = []
    labels = []

    for class_idx, cls in enumerate(CLASS_NAMES):
        cls_dir = root_dir / cls
        cls_files = []
        for ext in ["*.png", "*.jpg", "*.jpeg", "*.bmp", "*.tif", "*.tiff"]:
            cls_files.extend(cls_dir.glob(ext))
        cls_files = sorted(cls_files)
        image_files.extend(cls_files)
        labels.extend([class_idx] * len(cls_files))

    return image_files, np.asarray(labels, dtype=int)


_, y_val_multiclass = collect_multiclass_paths(VAL_DIR)
_, y_test_multiclass = collect_multiclass_paths(TEST_DIR)

print("Validation labels:", y_val_multiclass.shape)
print("Test labels:", y_test_multiclass.shape)


@torch.no_grad()
def predict_multiclass_probs(model, loader):
    model.eval()
    all_probs = []

    for images, _ in loader:
        images = images.to(device)
        logits = model(images)
        probs = torch.softmax(logits, dim=1)
        all_probs.append(probs.detach().cpu().numpy())

    return np.concatenate(all_probs, axis=0)


def evaluate_multiclass_predictions(y_true, y_pred, title):
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")

    print(f"================ {title} ================")
    print(f"Accuracy: {acc:.4f}")
    print(f"Macro-F1: {macro_f1:.4f}")
    print(classification_report(
        y_true,
        y_pred,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    ))

    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    print("Confusion matrix:")
    print(cm)

    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "y_pred": y_pred,
        "cm": cm,
    }


def extract_class_metrics(y_true, y_pred, class_idx):
    report = classification_report(
        y_true,
        y_pred,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
        output_dict=True,
    )

    class_name = CLASS_NAMES[class_idx]
    return {
        "Precision": report[class_name]["precision"],
        "Recall": report[class_name]["recall"],
        "F1": report[class_name]["f1-score"],
    }


Validation labels: (234,)
Test labels: (267,)


## 7. Four-model ensemble components

In [9]:

def first_valid_path(paths, model_name):
    for p in paths:
        p = Path(p)
        if p.exists() and p.stat().st_size > 1_000_000:
            with open(p, "rb") as f:
                first_bytes = f.read(80)
            if not first_bytes.startswith(b"version https://git-lfs.github.com"):
                print(f"{model_name}: {p}")
                return p
    raise FileNotFoundError(f"No valid checkpoint found for {model_name}")


SANDRA_MODELS = {
    "efficientnet": first_valid_path([
        "/kaggle/input/models/martimflix/efficientnet/pytorch/default/1/efficientnet_b7.pth",
        "/kaggle/input/models/martimflix/efficientnet-original/pytorch/default/1/efficientnet_b7_original.pth",
    ], "EfficientNet-B7"),

    "sandra_densenet": first_valid_path([
        "/kaggle/input/models/martimflix/densenet-sandra/pytorch/default/1/densenet_imagens_geradas_todas_410.pth",
        "/kaggle/input/models/martimflix/densenet-sandra-generated/pytorch/default/1/densenet_sandra.pth",
    ], "Sandra DenseNet"),

    "mobilenet": first_valid_path([
        "/kaggle/input/models/martimflix/mobilenet/pytorch/default/1/mobilenet_v2.pth",
        "/kaggle/input/models/martimflix/mobilenetv2/pytorch/default/1/mobilenet_v2.pth",
        "/kaggle/input/models/martimflix/mobilenet-original/pytorch/default/1/mobilenet_v2_original.pth",
    ], "MobileNetV2 original"),

    "resnet": first_valid_path([
        "/kaggle/input/models/martimflix/resnet/pytorch/default/1/resnet50_img_geradas_410.pth",
        "/kaggle/input/models/martimflix/resnet-sandra/pytorch/default/1/resnet50_sandra.pth",
        "/kaggle/input/models/martimflix/resnet/pytorch/default/1/resnet50.pth",
    ], "ResNet50"),
}

RUNNING_ON_KAGGLE = Path("/kaggle/input").exists()

if RUNNING_ON_KAGGLE:
    expected_paths = {
        "efficientnet": "/kaggle/input/models/martimflix/efficientnet/pytorch/default/1/efficientnet_b7.pth",
        "sandra_densenet": "/kaggle/input/models/martimflix/densenet-sandra/pytorch/default/1/densenet_imagens_geradas_todas_410.pth",
        "mobilenet": "/kaggle/input/models/martimflix/mobilenet/pytorch/default/1/mobilenet_v2.pth",
        "resnet": "/kaggle/input/models/martimflix/resnet/pytorch/default/1/resnet50_img_geradas_410.pth",
    }

    for model_name, expected_path in expected_paths.items():
        actual_path = str(SANDRA_MODELS[model_name])

        assert actual_path == expected_path, (
            f"Wrong checkpoint selected for {model_name}.\n"
            f"Expected: {expected_path}\n"
            f"Got:      {actual_path}\n"
            "This may change the final reproduced result."
        )

    print("Kaggle checkpoint validation passed: exact validated checkpoints are being used.")

else:
    print("Local execution detected: skipping absolute Kaggle path validation.")

    for model_name, path in SANDRA_MODELS.items():
        path = Path(path)

        assert path.exists(), f"Checkpoint not found for {model_name}: {path}"
        assert path.stat().st_size > 1_000_000, (
            f"Checkpoint for {model_name} is too small and may be invalid: {path}"
        )

        with open(path, "rb") as f:
            first_bytes = f.read(80)

        assert not first_bytes.startswith(b"version https://git-lfs.github.com"), (
            f"Checkpoint for {model_name} is a Git LFS pointer, not a real model: {path}"
        )

    print("Local checkpoint validation passed: selected files look like valid .pth checkpoints.")


pd.DataFrame([
    {
        "model": name,
        "path": str(path),
        "size_mb": path.stat().st_size / (1024 ** 2),
    }
    for name, path in SANDRA_MODELS.items()
])


def create_ensemble_model(arch):
    if arch == "efficientnet":
        model = efficientnet_b7(weights=None)
        model.classifier[1] = nn.Linear(
            model.classifier[1].in_features,
            NUM_CLASSES,
        )

    elif arch == "sandra_densenet":
        model = MonaiDenseNet121(
            spatial_dims=2,
            in_channels=3,
            out_channels=NUM_CLASSES,
        )

    elif arch == "mobilenet":
        model = mobilenet_v2(weights=None)
        model.classifier[1] = nn.Linear(
            model.classifier[1].in_features,
            NUM_CLASSES,
        )

    elif arch == "resnet":
        model = resnet50(weights=None)
        model.fc = nn.Linear(
            model.fc.in_features,
            NUM_CLASSES,
        )

    else:
        raise ValueError(f"Unknown architecture: {arch}")

    model = load_checkpoint_safely(model, SANDRA_MODELS[arch])
    model = model.to(device)
    model.eval()
    return model


sandra_individual_val_probs = {}
sandra_individual_test_probs = {}
individual_rows = []

for arch in SANDRA_MODELS:
    print("=" * 100)
    print(f"Loading {arch}...")

    model_tmp = create_ensemble_model(arch)

    val_probs = predict_multiclass_probs(model_tmp, binary_val_loader)
    test_probs = predict_multiclass_probs(model_tmp, binary_test_loader)

    sandra_individual_val_probs[arch] = val_probs
    sandra_individual_test_probs[arch] = test_probs

    val_preds = np.argmax(val_probs, axis=1)
    test_preds = np.argmax(test_probs, axis=1)

    individual_rows.append({
        "Model": arch,
        "Val Macro-F1": f1_score(y_val_multiclass, val_preds, average="macro"),
        "Test Macro-F1": f1_score(y_test_multiclass, test_preds, average="macro"),
    })

    del model_tmp
    if device.type == "cuda":
        torch.cuda.empty_cache()

individual_model_results = pd.DataFrame(individual_rows).sort_values(
    by="Val Macro-F1",
    ascending=False,
).reset_index(drop=True)

individual_model_results.round(4)


EfficientNet-B7: /kaggle/input/models/martimflix/efficientnet/pytorch/default/1/efficientnet_b7.pth
Sandra DenseNet: /kaggle/input/models/martimflix/densenet-sandra/pytorch/default/1/densenet_imagens_geradas_todas_410.pth
MobileNetV2 original: /kaggle/input/models/martimflix/mobilenet/pytorch/default/1/mobilenet_v2.pth
ResNet50: /kaggle/input/models/martimflix/resnet/pytorch/default/1/resnet50_img_geradas_410.pth
Kaggle checkpoint validation passed: exact validated checkpoints are being used.
Loading efficientnet...
Checkpoint loaded.
Missing keys: 0
Unexpected keys: 0
Loading sandra_densenet...
Checkpoint loaded.
Missing keys: 0
Unexpected keys: 0
Loading mobilenet...
Checkpoint loaded.
Missing keys: 0
Unexpected keys: 0
Loading resnet...
Checkpoint loaded.
Missing keys: 0
Unexpected keys: 0


,Model,Val Macro-F1,Test Macro-F1
0,efficientnet,0.6008,0.7439
1,mobilenet,0.5893,0.5578
2,sandra_densenet,0.5825,0.6665
3,resnet,0.5688,0.7359


## 8. EfficientNet-B0 

In [10]:
# EfficientNet-B0 VAE

from monai.transforms import Compose, LoadImage, EnsureChannelFirst, Resize, NormalizeIntensity, Lambda, ToTensor


EFFICIENTNET_B0_CKPT_PATH = Path(
    "/kaggle/input/models/martimflix/efficient-netb0/pytorch/default/1/efficientnet_b0_VAE_resized.pth"
)

assert EFFICIENTNET_B0_CKPT_PATH.exists(), f"EfficientNet-B0 checkpoint not found: {EFFICIENTNET_B0_CKPT_PATH}"
assert EFFICIENTNET_B0_CKPT_PATH.stat().st_size > 1_000_000, "EfficientNet-B0 checkpoint seems invalid."

print("EfficientNet-B0 checkpoint:")
print(EFFICIENTNET_B0_CKPT_PATH)
print("Size MB:", EFFICIENTNET_B0_CKPT_PATH.stat().st_size / (1024 ** 2))


# EfficientNet-B0 notebook used 224x224 images.
b0_eval_transforms = Compose([
    LoadImage(image_only=True),
    EnsureChannelFirst(),
    Resize((224, 224)),
    NormalizeIntensity(),
    Lambda(repeat_if_needed),
    ToTensor()
])


class MulticlassPathDataset(Dataset):
    def __init__(self, image_files, labels, transform):
        self.image_files = list(image_files)
        self.labels = list(labels)
        self.transform = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img = self.transform(str(self.image_files[idx]))
        label = int(self.labels[idx])
        return img, label


val_files_multiclass, y_val_check = collect_multiclass_paths(VAL_DIR)
test_files_multiclass, y_test_check = collect_multiclass_paths(TEST_DIR)

assert np.array_equal(y_val_check, y_val_multiclass), "Validation label order mismatch."
assert np.array_equal(y_test_check, y_test_multiclass), "Test label order mismatch."


b0_val_ds = MulticlassPathDataset(
    val_files_multiclass,
    y_val_multiclass,
    b0_eval_transforms
)

b0_test_ds = MulticlassPathDataset(
    test_files_multiclass,
    y_test_multiclass,
    b0_eval_transforms
)

b0_val_loader = DataLoader(
    b0_val_ds,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

b0_test_loader = DataLoader(
    b0_test_ds,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)


def create_efficientnet_b0_vae_model():
    model = efficientnet_b0(weights=None)

    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
    )

    model = load_checkpoint_safely(model, EFFICIENTNET_B0_CKPT_PATH)
    model = model.to(device)
    model.eval()
    return model


efficientnet_b0_model = create_efficientnet_b0_vae_model()

b0_val_probs = predict_multiclass_probs(efficientnet_b0_model, b0_val_loader)
b0_test_probs = predict_multiclass_probs(efficientnet_b0_model, b0_test_loader)

b0_val_preds = np.argmax(b0_val_probs, axis=1)
b0_test_preds = np.argmax(b0_test_probs, axis=1)

b0_val_eval = evaluate_multiclass_predictions(
    y_val_multiclass,
    b0_val_preds,
    title="EfficientNet-B0 VAE - Validation"
)

b0_test_eval = evaluate_multiclass_predictions(
    y_test_multiclass,
    b0_test_preds,
    title="EfficientNet-B0 VAE - Test"
)

del efficientnet_b0_model
if device.type == "cuda":
    torch.cuda.empty_cache()

EfficientNet-B0 checkpoint:
/kaggle/input/models/martimflix/efficient-netb0/pytorch/default/1/efficientnet_b0_VAE_resized.pth
Size MB: 15.598944664001465
Checkpoint loaded.
Missing keys: 0
Unexpected keys: 0
================ EfficientNet-B0 VAE - Validation ================
Accuracy: 0.5726
Macro-F1: 0.5085
               precision    recall  f1-score   support

Biliary_Leaks     0.3846    0.2083    0.2703        24
    Lithiasis     0.7381    0.6327    0.6813        98
       Normal     0.6038    0.5424    0.5714        59
    Stricture     0.4167    0.6604    0.5109        53

     accuracy                         0.5726       234
    macro avg     0.5358    0.5109    0.5085       234
 weighted avg     0.5952    0.5726    0.5729       234

Confusion matrix:
[[ 5  5 10  4]
 [ 2 62  8 26]
 [ 1  7 32 19]
 [ 5 10  3 35]]
================ EfficientNet-B0 VAE - Test ================
Accuracy: 0.6966
Macro-F1: 0.6492
               precision    recall  f1-score   support

Biliary_Leaks     

## 9. ConvNeXt-Tiny 

In [ ]:
# ConvNeXt-Tiny 

from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
import seaborn as sns
import torch.nn.functional as F
from IPython.display import FileLink, display
import shutil
import json


def find_convnext_checkpoint():
    candidate_paths = [
        "/kaggle/input/models/martimflix/convenet-sandra/pytorch/default/1/convnext_tiny_VAE_seed0.pth",
        "/kaggle/input/models/martimflix/convnet/pytorch/default/1/convnext_tiny_VAE_seed0.pth",
        "/kaggle/input/models/martimflix/convNet/pytorch/default/1/convnext_tiny_VAE_seed0.pth",
        "/kaggle/input/convenet/pytorch/default/1/convnext_tiny_VAE_seed0.pth",
        "/kaggle/input/convnet/pytorch/default/1/convnext_tiny_VAE_seed0.pth",
        "/kaggle/input/convNet/pytorch/default/1/convnext_tiny_VAE_seed0.pth",
    ]

    for p in candidate_paths:
        p = Path(p)
        if p.exists() and p.stat().st_size > 1_000_000:
            with open(p, "rb") as f:
                first_bytes = f.read(80)
            if not first_bytes.startswith(b"version https://git-lfs.github.com"):
                print("ConvNeXt checkpoint found:")
                print(p)
                print("Size MB:", p.stat().st_size / (1024 ** 2))
                return p

    print("Searching for ConvNeXt checkpoint inside /kaggle/input ...")
    matches = []
    for p in Path("/kaggle/input").rglob("*.pth"):
        name = p.name.lower()
        path_str = str(p).lower()
        if (
            ("convnext" in name or "convnext" in path_str or "convenet" in path_str or "convnet" in path_str)
            and p.stat().st_size > 1_000_000
        ):
            with open(p, "rb") as f:
                first_bytes = f.read(80)
            if not first_bytes.startswith(b"version https://git-lfs.github.com"):
                matches.append(p)

    if len(matches) == 0:
        print("All .pth files under /kaggle/input:")
        for p in Path("/kaggle/input").rglob("*.pth"):
            print("-", p, "| size MB:", p.stat().st_size / (1024 ** 2))
        raise FileNotFoundError("Could not find convnext_tiny_VAE_seed0.pth. Attach the Kaggle model/input first.")

    print("Possible ConvNeXt checkpoints found:")
    for p in matches:
        print("-", p, "| size MB:", p.stat().st_size / (1024 ** 2))

    return matches[0]


CONVNEXT_CKPT_PATH = find_convnext_checkpoint()


def create_convnext_tiny_vae_model(strict=True):
    """
    Creates and loads the ConvNeXt-Tiny VAE model using the same head used during training.
    strict=True is intentionally used to ensure the checkpoint matches the architecture exactly.
    """
    model = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)

    in_features = model.classifier[2].in_features
    model.classifier[2] = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, NUM_CLASSES),
    )

    state_dict = torch.load(CONVNEXT_CKPT_PATH, map_location=device)
    model.load_state_dict(state_dict, strict=strict)

    model = model.to(device)
    model.eval()
    return model


convnext_model = create_convnext_tiny_vae_model(strict=True)
print("ConvNeXt-Tiny VAE loaded with strict=True.")


ConvNeXt checkpoint found:
/kaggle/input/models/martimflix/convenet-sandra/pytorch/default/1/convnext_tiny_VAE_seed0.pth
Size MB: 106.20369243621826
Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 208MB/s]  


ConvNeXt-Tiny VAE loaded with strict=True.


In [12]:
# ConvNeXt probabilities

convnext_val_probs = predict_multiclass_probs(convnext_model, binary_val_loader)
convnext_test_probs = predict_multiclass_probs(convnext_model, binary_test_loader)

convnext_val_preds = np.argmax(convnext_val_probs, axis=1)
convnext_test_preds = np.argmax(convnext_test_probs, axis=1)

display(pd.DataFrame([
    {
        "Model": "ConvNeXt-Tiny",
        "Split": "validation",
        "Accuracy": accuracy_score(y_val_multiclass, convnext_val_preds),
        "Macro-F1": f1_score(y_val_multiclass, convnext_val_preds, average="macro"),
    },
    {
        "Model": "ConvNeXt-Tiny",
        "Split": "test",
        "Accuracy": accuracy_score(y_test_multiclass, convnext_test_preds),
        "Macro-F1": f1_score(y_test_multiclass, convnext_test_preds, average="macro"),
    },
]).round(4))


,Model,Split,Accuracy,Macro-F1
0,ConvNeXt-Tiny,validation,0.6838,0.6594
1,ConvNeXt-Tiny,test,0.7715,0.7434


## 10. Build fixed base blend

In [13]:

# Fixed base blend: current final hierarchical blend without Biliary override

JOINT4_W_EFF = 0.24
JOINT4_W_DENSE = 0.21
JOINT4_W_MOBILE = 0.35
JOINT4_W_RESNET = 0.20
BEST_CONVNEXT_ALPHA = 0.32
FINAL_ALPHA_B0 = 0.08

joint4_val_probs = (
    JOINT4_W_EFF * sandra_individual_val_probs["efficientnet"]
    + JOINT4_W_DENSE * sandra_individual_val_probs["sandra_densenet"]
    + JOINT4_W_MOBILE * sandra_individual_val_probs["mobilenet"]
    + JOINT4_W_RESNET * sandra_individual_val_probs["resnet"]
)

joint4_test_probs = (
    JOINT4_W_EFF * sandra_individual_test_probs["efficientnet"]
    + JOINT4_W_DENSE * sandra_individual_test_probs["sandra_densenet"]
    + JOINT4_W_MOBILE * sandra_individual_test_probs["mobilenet"]
    + JOINT4_W_RESNET * sandra_individual_test_probs["resnet"]
)

convnext_blend_val_probs = (
    (1.0 - BEST_CONVNEXT_ALPHA) * joint4_val_probs
    + BEST_CONVNEXT_ALPHA * convnext_val_probs
)

convnext_blend_test_probs = (
    (1.0 - BEST_CONVNEXT_ALPHA) * joint4_test_probs
    + BEST_CONVNEXT_ALPHA * convnext_test_probs
)

base_val_probs = (
    (1.0 - FINAL_ALPHA_B0) * convnext_blend_val_probs
    + FINAL_ALPHA_B0 * b0_val_probs
)

base_test_probs = (
    (1.0 - FINAL_ALPHA_B0) * convnext_blend_test_probs
    + FINAL_ALPHA_B0 * b0_test_probs
)

base_val_preds = np.argmax(base_val_probs, axis=1)
base_test_preds = np.argmax(base_test_probs, axis=1)

base_val_eval = evaluate_multiclass_predictions(
    y_val_multiclass,
    base_val_preds,
    title="Base hierarchical blend - Validation",
)

base_test_eval = evaluate_multiclass_predictions(
    y_test_multiclass,
    base_test_preds,
    title="Base hierarchical blend - Test",
)


================ Base hierarchical blend - Validation ================
Accuracy: 0.7308
Macro-F1: 0.6942
               precision    recall  f1-score   support

Biliary_Leaks     0.8462    0.4583    0.5946        24
    Lithiasis     0.7767    0.8163    0.7960        98
       Normal     0.7000    0.5932    0.6422        59
    Stricture     0.6618    0.8491    0.7438        53

     accuracy                         0.7308       234
    macro avg     0.7462    0.6792    0.6942       234
 weighted avg     0.7385    0.7308    0.7248       234

Confusion matrix:
[[11  5  5  3]
 [ 1 80  6 11]
 [ 1 14 35  9]
 [ 0  4  4 45]]
================ Base hierarchical blend - Test ================
Accuracy: 0.8127
Macro-F1: 0.7868
               precision    recall  f1-score   support

Biliary_Leaks     0.8462    0.6471    0.7333        17
    Lithiasis     0.8226    0.8293    0.8259       123
       Normal     0.6429    0.8372    0.7273        43
    Stricture     0.9189    0.8095    0.8608        8

## 11. Stricture vs Lithiasis specialist data

In [14]:

# Stricture vs Lithiasis specialist data

LITHIASIS_CLASS = "Lithiasis"
STRICTURE_CLASS = "Stricture"
LITHIASIS_IDX = CLASS_NAMES.index(LITHIASIS_CLASS)
STRICTURE_IDX = CLASS_NAMES.index(STRICTURE_CLASS)

# Binary target for this specialist:
# 0 = Lithiasis
# 1 = Stricture
PAIR_CLASS_NAMES = [LITHIASIS_CLASS, STRICTURE_CLASS]


def collect_stricture_lithiasis_paths(root_dir):
    image_files = []
    labels = []

    for cls, binary_label in [(LITHIASIS_CLASS, 0), (STRICTURE_CLASS, 1)]:
        cls_dir = root_dir / cls
        cls_files = []
        for ext in ["*.png", "*.jpg", "*.jpeg", "*.bmp", "*.tif", "*.tiff"]:
            cls_files.extend(cls_dir.glob(ext))
        cls_files = sorted(cls_files)
        image_files.extend(cls_files)
        labels.extend([binary_label] * len(cls_files))

    return image_files, labels


class PairPathDataset(Dataset):
    def __init__(self, image_files, labels, transform):
        self.image_files = list(image_files)
        self.labels = list(labels)
        self.transform = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img = self.transform(str(self.image_files[idx]))
        label = torch.tensor(float(self.labels[idx]), dtype=torch.float32)
        return img, label


pair_train_files, pair_train_labels = collect_stricture_lithiasis_paths(REAL_TRAIN_DIR)
pair_val_files, pair_val_labels = collect_stricture_lithiasis_paths(VAL_DIR)
pair_test_files, pair_test_labels = collect_stricture_lithiasis_paths(TEST_DIR)

pair_train_ds = PairPathDataset(pair_train_files, pair_train_labels, train_transforms)
pair_val_ds = PairPathDataset(pair_val_files, pair_val_labels, eval_transforms)
pair_test_ds = PairPathDataset(pair_test_files, pair_test_labels, eval_transforms)

pair_train_loader = DataLoader(pair_train_ds, batch_size=8, shuffle=True, num_workers=2, pin_memory=(device.type == "cuda"))
pair_val_loader = DataLoader(pair_val_ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=(device.type == "cuda"))
pair_test_loader = DataLoader(pair_test_ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=(device.type == "cuda"))

pd.DataFrame([
    {"Split": "Train", "Lithiasis": int(np.sum(np.array(pair_train_labels) == 0)), "Stricture": int(np.sum(np.array(pair_train_labels) == 1)), "Total": len(pair_train_labels)},
    {"Split": "Validation", "Lithiasis": int(np.sum(np.array(pair_val_labels) == 0)), "Stricture": int(np.sum(np.array(pair_val_labels) == 1)), "Total": len(pair_val_labels)},
    {"Split": "Test", "Lithiasis": int(np.sum(np.array(pair_test_labels) == 0)), "Stricture": int(np.sum(np.array(pair_test_labels) == 1)), "Total": len(pair_test_labels)},
])


,Split,Lithiasis,Stricture,Total
0,Train,505,255,760
1,Validation,98,53,151
2,Test,123,84,207


## 12. Train Stricture vs Lithiasis specialist

In [15]:

# Create Stricture vs Lithiasis specialist

def amp_context():
    if device.type == "cuda":
        return torch.cuda.amp.autocast(enabled=True)
    return nullcontext()


def create_pair_efficientnet_b7():
    model = efficientnet_b7(weights=None)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
    model = load_checkpoint_safely(model, EFFICIENTNET_CKPT_PATH)

    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, 1)
    model = model.to(device)
    return model


pair_model = create_pair_efficientnet_b7()

# Freeze first, then fine-tune classifier + last feature blocks.
for p in pair_model.parameters():
    p.requires_grad = False

for p in pair_model.classifier.parameters():
    p.requires_grad = True

# Partial fine-tuning: last EfficientNet-B7 blocks.
for p in pair_model.features[-2:].parameters():
    p.requires_grad = True

trainable_params = sum(p.numel() for p in pair_model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in pair_model.parameters())
print(f"Trainable params: {trainable_params:,}")
print(f"Total params:     {total_params:,}")
print(f"Trainable ratio:  {100 * trainable_params / total_params:.4f}%")

n_pos = int(sum(pair_train_labels))       # Stricture
n_neg = int(len(pair_train_labels) - n_pos)  # Lithiasis
pos_weight_value = n_neg / max(n_pos, 1)
pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=device)

print("Lithiasis samples:", n_neg)
print("Stricture samples:", n_pos)
print("pos_weight:", pos_weight_value)

pair_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

pair_optimizer = torch.optim.AdamW(
    [
        {"params": pair_model.features[-2:].parameters(), "lr": 1e-5, "weight_decay": 1e-4},
        {"params": pair_model.classifier.parameters(), "lr": 1e-4, "weight_decay": 1e-4},
    ]
)

pair_scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))


Checkpoint loaded.
Missing keys: 0
Unexpected keys: 0
Trainable params: 23,078,977
Total params:     63,789,521
Trainable ratio:  36.1799%
Lithiasis samples: 505
Stricture samples: 255
pos_weight: 1.9803921568627452


/tmp/ipykernel_58/2078533576.py:57: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  pair_scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))


In [16]:

# Pair specialist utilities

@torch.no_grad()
def predict_pair_probs(model, loader):
    model.eval()
    all_probs = []
    all_labels = []

    for images, labels in loader:
        images = images.to(device)
        logits = model(images).reshape(-1)
        probs = torch.sigmoid(logits)
        all_probs.append(probs.detach().cpu().numpy())
        all_labels.append(labels.detach().cpu().numpy())

    return np.concatenate(all_labels).astype(int), np.concatenate(all_probs)


def evaluate_pair_probs(probs, y_true, threshold, title):
    preds = (probs >= threshold).astype(int)
    print(f"================ {title} ================")
    print(f"Threshold: {threshold:.3f}")
    print(f"Accuracy:  {accuracy_score(y_true, preds):.4f}")
    print(f"F1 Stricture: {f1_score(y_true, preds, pos_label=1, zero_division=0):.4f}")
    print(classification_report(y_true, preds, target_names=PAIR_CLASS_NAMES, digits=4, zero_division=0))
    print("Confusion matrix [Lithiasis, Stricture]:")
    print(confusion_matrix(y_true, preds, labels=[0, 1]))
    return {
        "accuracy": accuracy_score(y_true, preds),
        "f1_stricture": f1_score(y_true, preds, pos_label=1, zero_division=0),
        "preds": preds,
    }


def evaluate_pair_loss(model, loader):
    model.eval()
    total_loss = 0.0
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            logits = model(images).reshape(-1)
            loss = pair_criterion(logits, labels)
            total_loss += loss.item() * images.size(0)
            all_probs.append(torch.sigmoid(logits).detach().cpu().numpy())
            all_labels.append(labels.detach().cpu().numpy())

    probs = np.concatenate(all_probs)
    y_true = np.concatenate(all_labels).astype(int)

    best_f1 = -1.0
    best_threshold = 0.5
    for threshold in np.arange(0.10, 0.91, 0.01):
        preds = (probs >= threshold).astype(int)
        f1 = f1_score(y_true, preds, pos_label=1, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = float(threshold)

    return {
        "loss": total_loss / len(loader.dataset),
        "best_f1": best_f1,
        "best_threshold": best_threshold,
    }


def set_pair_training_mode(model):
    model.train()
    # Frozen layers remain frozen due to requires_grad=False.


In [17]:

# Train pair specialist with validation early stopping

PAIR_EPOCHS = 25
PAIR_PATIENCE = 5
PAIR_BEST_MODEL_PATH = "/kaggle/working/efficientnet_b7_stricture_lithiasis_specialist_best.pth"

best_val_f1 = -1.0
best_epoch = -1
best_pair_threshold = 0.5
patience_counter = 0
pair_history = []

for epoch in range(1, PAIR_EPOCHS + 1):
    start_time = time.time()
    set_pair_training_mode(pair_model)

    train_loss = 0.0
    for images, labels in pair_train_loader:
        images = images.to(device)
        labels = labels.to(device)

        pair_optimizer.zero_grad(set_to_none=True)

        with amp_context():
            logits = pair_model(images).reshape(-1)
            loss = pair_criterion(logits, labels)

        pair_scaler.scale(loss).backward()
        pair_scaler.step(pair_optimizer)
        pair_scaler.update()

        train_loss += loss.item() * images.size(0)

    train_loss /= len(pair_train_loader.dataset)
    val_eval = evaluate_pair_loss(pair_model, pair_val_loader)
    elapsed = time.time() - start_time

    pair_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_eval["loss"],
        "val_best_f1": val_eval["best_f1"],
        "val_best_threshold": val_eval["best_threshold"],
        "time_sec": elapsed,
    })

    print(
        f"Epoch {epoch:02d}/{PAIR_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_eval['loss']:.4f} | "
        f"Val F1: {val_eval['best_f1']:.4f} | "
        f"Best thr: {val_eval['best_threshold']:.2f} | "
        f"Time: {elapsed:.1f}s"
    )

    if val_eval["best_f1"] > best_val_f1:
        best_val_f1 = val_eval["best_f1"]
        best_pair_threshold = float(val_eval["best_threshold"])
        best_epoch = epoch
        patience_counter = 0
        torch.save(copy.deepcopy(pair_model.state_dict()), PAIR_BEST_MODEL_PATH)
        print("Saved new best pair specialist.")
    else:
        patience_counter += 1
        if patience_counter >= PAIR_PATIENCE:
            print("Early stopping.")
            break

pair_history_df = pd.DataFrame(pair_history)
display(pair_history_df)
print("Best epoch:", best_epoch)
print("Best validation pair F1:", best_val_f1)
print("Best pair threshold:", best_pair_threshold)


/tmp/ipykernel_58/2078533576.py:5: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast(enabled=True)


Epoch 01/25 | Train Loss: 0.4599 | Val Loss: 0.5919 | Val F1: 0.7333 | Best thr: 0.39 | Time: 48.2s
Saved new best pair specialist.


/tmp/ipykernel_58/2078533576.py:5: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast(enabled=True)


Epoch 02/25 | Train Loss: 0.1135 | Val Loss: 0.6609 | Val F1: 0.7273 | Best thr: 0.29 | Time: 29.6s


/tmp/ipykernel_58/2078533576.py:5: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast(enabled=True)


...

Epoch 05/25 | Train Loss: 0.0283 | Val Loss: 0.8452 | Val F1: 0.7302 | Best thr: 0.12 | Time: 28.9s


/tmp/ipykernel_58/2078533576.py:5: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast(enabled=True)


Epoch 06/25 | Train Loss: 0.0161 | Val Loss: 0.9009 | Val F1: 0.7302 | Best thr: 0.12 | Time: 28.5s
Early stopping.


,epoch,train_loss,val_loss,val_best_f1,val_best_threshold,time_sec
0,1,0.459860,0.591877,0.733333,0.39,48.247680
1,2,0.113464,0.660890,0.727273,0.29,29.647759
2,3,0.055436,0.727879,0.721311,0.23,28.109154
3,4,0.035949,0.802971,0.721311,0.18,28.509722
4,5,0.028316,0.845159,0.730159,0.12,28.857973
5,6,0.016148,0.900946,0.730159,0.12,28.534749


Best epoch: 1
Best validation pair F1: 0.7333333333333333
Best pair threshold: 0.3899999999999999


## 13. Apply Stricture/Lithiasis specialist as conservative override

In [18]:

# Pair specialist probabilities

pair_model.load_state_dict(torch.load(PAIR_BEST_MODEL_PATH, map_location=device))
pair_model = pair_model.to(device)
pair_model.eval()

pair_y_val, pair_val_probs = predict_pair_probs(pair_model, pair_val_loader)
pair_y_test, pair_test_probs = predict_pair_probs(pair_model, pair_test_loader)

_ = evaluate_pair_probs(
    pair_val_probs,
    pair_y_val,
    threshold=best_pair_threshold,
    title="Stricture vs Lithiasis specialist - Validation",
)

_ = evaluate_pair_probs(
    pair_test_probs,
    pair_y_test,
    threshold=best_pair_threshold,
    title="Stricture vs Lithiasis specialist - Test",
)


================ Stricture vs Lithiasis specialist - Validation ================
Threshold: 0.390
Accuracy:  0.7881
F1 Stricture: 0.7333
              precision    recall  f1-score   support

   Lithiasis     0.8929    0.7653    0.8242        98
   Stricture     0.6567    0.8302    0.7333        53

    accuracy                         0.7881       151
   macro avg     0.7748    0.7977    0.7788       151
weighted avg     0.8100    0.7881    0.7923       151

Confusion matrix [Lithiasis, Stricture]:
[[75 23]
 [ 9 44]]
================ Stricture vs Lithiasis specialist - Test ================
Threshold: 0.390
Accuracy:  0.8261
F1 Stricture: 0.8065
              precision    recall  f1-score   support

   Lithiasis     0.9143    0.7805    0.8421       123
   Stricture     0.7353    0.8929    0.8065        84

    accuracy                         0.8261       207
   macro avg     0.8248    0.8367    0.8243       207
weighted avg     0.8417    0.8261    0.8276       207

Confusion matrix [

In [19]:

# Map pair probabilities back to full validation/test order

def build_pair_prob_lookup(pair_files, pair_probs):
    return {str(path): float(prob) for path, prob in zip(pair_files, pair_probs)}

val_pair_lookup = build_pair_prob_lookup(pair_val_files, pair_val_probs)
test_pair_lookup = build_pair_prob_lookup(pair_test_files, pair_test_probs)

val_files_full, _ = collect_multiclass_paths(VAL_DIR)
test_files_full, _ = collect_multiclass_paths(TEST_DIR)

val_pair_probs_full = np.full(len(val_files_full), np.nan, dtype=float)
test_pair_probs_full = np.full(len(test_files_full), np.nan, dtype=float)

for i, path in enumerate(val_files_full):
    val_pair_probs_full[i] = val_pair_lookup.get(str(path), np.nan)

for i, path in enumerate(test_files_full):
    test_pair_probs_full[i] = test_pair_lookup.get(str(path), np.nan)

print("Validation pair probabilities available:", np.sum(~np.isnan(val_pair_probs_full)))
print("Test pair probabilities available:", np.sum(~np.isnan(test_pair_probs_full)))


Validation pair probabilities available: 151
Test pair probabilities available: 207


In [20]:

# Conservative override threshold search on validation

def apply_pair_override(base_preds, pair_probs_full, tau):
    preds = base_preds.copy()
    candidates = np.isin(base_preds, [LITHIASIS_IDX, STRICTURE_IDX]) & ~np.isnan(pair_probs_full)

    to_stricture = candidates & (pair_probs_full >= tau)
    to_lithiasis = candidates & (pair_probs_full <= (1.0 - tau))

    preds[to_stricture] = STRICTURE_IDX
    preds[to_lithiasis] = LITHIASIS_IDX

    changed = preds != base_preds
    return preds, changed


rows = []
for tau in np.arange(0.50, 0.96, 0.01):
    val_preds_tau, val_changed_tau = apply_pair_override(base_val_preds, val_pair_probs_full, tau)
    test_preds_tau, test_changed_tau = apply_pair_override(base_test_preds, test_pair_probs_full, tau)

    rows.append({
        "tau": float(tau),
        "val_macro_f1": f1_score(y_val_multiclass, val_preds_tau, average="macro"),
        "val_accuracy": accuracy_score(y_val_multiclass, val_preds_tau),
        "val_n_changed": int(val_changed_tau.sum()),
        "test_macro_f1": f1_score(y_test_multiclass, test_preds_tau, average="macro"),
        "test_accuracy": accuracy_score(y_test_multiclass, test_preds_tau),
        "test_n_changed": int(test_changed_tau.sum()),
    })

pair_override_search_df = pd.DataFrame(rows).sort_values(
    by=["val_macro_f1", "val_accuracy"],
    ascending=False,
).reset_index(drop=True)

display(pair_override_search_df.head(20).round(4))

BEST_PAIR_OVERRIDE_TAU = float(pair_override_search_df.iloc[0]["tau"])
print("Selected override confidence tau:", BEST_PAIR_OVERRIDE_TAU)


,tau,val_macro_f1,val_accuracy,val_n_changed,test_macro_f1,test_accuracy,test_n_changed
0,0.93,0.6942,0.7308,0,0.7868,0.8127,0
1,0.94,0.6942,0.7308,0,0.7868,0.8127,0
2,0.95,0.6942,0.7308,0,0.7868,0.8127,0
3,0.82,0.6911,0.7265,1,0.7894,0.8165,1
4,0.83,0.6911,0.7265,1,0.7894,0.8165,1
5,0.84,0.6911,0.7265,1,0.7868,0.8127,0
6,0.85,0.6911,0.7265,1,0.7868,0.8127,0
7,0.86,0.6911,0.7265,1,0.7868,0.8127,0
8,0.87,0.6911,0.7265,1,0.7868,0.8127,0
9,0.88,0.6911,0.7265,1,0.7868,0.8127,0


Selected override confidence tau: 0.9300000000000004


In [21]:
# Final evaluation with Stricture/Lithiasis specialist

final_pair_val_preds, val_changed = apply_pair_override(
    base_val_preds,
    val_pair_probs_full,
    BEST_PAIR_OVERRIDE_TAU,
)

final_pair_test_preds, test_changed = apply_pair_override(
    base_test_preds,
    test_pair_probs_full,
    BEST_PAIR_OVERRIDE_TAU,
)

final_pair_val_eval = evaluate_multiclass_predictions(
    y_val_multiclass,
    final_pair_val_preds,
    title="Base blend + Stricture/Lithiasis specialist - Validation",
)

final_pair_test_eval = evaluate_multiclass_predictions(
    y_test_multiclass,
    final_pair_test_preds,
    title="Base blend + Stricture/Lithiasis specialist - Test",
)

pair_specialist_comparison_df = pd.DataFrame([
    {
        "System": "Base hierarchical blend",
        "Val Macro-F1": base_val_eval["macro_f1"],
        "Test Macro-F1": base_test_eval["macro_f1"],
        "Val Accuracy": base_val_eval["accuracy"],
        "Test Accuracy": base_test_eval["accuracy"],
        "Changed test decisions": 0,
    },
    {
        "System": "Base blend + Stricture/Lithiasis specialist",
        "Val Macro-F1": final_pair_val_eval["macro_f1"],
        "Test Macro-F1": final_pair_test_eval["macro_f1"],
        "Val Accuracy": final_pair_val_eval["accuracy"],
        "Test Accuracy": final_pair_test_eval["accuracy"],
        "Changed test decisions": int(test_changed.sum()),
    },
])

display(pair_specialist_comparison_df.round(4))


================ Base blend + Stricture/Lithiasis specialist - Validation ================
Accuracy: 0.7308
Macro-F1: 0.6942
               precision    recall  f1-score   support

Biliary_Leaks     0.8462    0.4583    0.5946        24
    Lithiasis     0.7767    0.8163    0.7960        98
       Normal     0.7000    0.5932    0.6422        59
    Stricture     0.6618    0.8491    0.7438        53

     accuracy                         0.7308       234
    macro avg     0.7462    0.6792    0.6942       234
 weighted avg     0.7385    0.7308    0.7248       234

Confusion matrix:
[[11  5  5  3]
 [ 1 80  6 11]
 [ 1 14 35  9]
 [ 0  4  4 45]]
================ Base blend + Stricture/Lithiasis specialist - Test ================
Accuracy: 0.8127
Macro-F1: 0.7868
               precision    recall  f1-score   support

Biliary_Leaks     0.8462    0.6471    0.7333        17
    Lithiasis     0.8226    0.8293    0.8259       123
       Normal     0.6429    0.8372    0.7273        43
    Stricture

,System,Val Macro-F1,Test Macro-F1,Val Accuracy,Test Accuracy,Changed test decisions
0,Base hierarchical blend,0.6942,0.7868,0.7308,0.8127,0
1,Base blend + Stricture/Lithiasis specialist,0.6942,0.7868,0.7308,0.8127,0


## 14. Impact analysis and save outputs

In [22]:
# Impact analysis for changed decisions


PAIR_OUTPUT_DIR = Path("/kaggle/working/stricture_lithiasis_specialist")
PAIR_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

rows = []
for idx, path in enumerate(test_files_full):
    if not test_changed[idx]:
        continue

    true_idx = int(y_test_multiclass[idx])
    base_idx = int(base_test_preds[idx])
    final_idx = int(final_pair_test_preds[idx])

    base_correct = base_idx == true_idx
    final_correct = final_idx == true_idx

    if (not base_correct) and final_correct:
        effect = "corrected_error"
    elif base_correct and (not final_correct):
        effect = "introduced_error"
    elif (not base_correct) and (not final_correct):
        effect = "changed_but_still_wrong"
    else:
        effect = "changed_but_still_correct"

    rows.append({
        "idx": idx,
        "filename": Path(path).name,
        "path": str(path),
        "true_class": CLASS_NAMES[true_idx],
        "base_prediction": CLASS_NAMES[base_idx],
        "final_prediction": CLASS_NAMES[final_idx],
        "effect": effect,
        "pair_probability_stricture": float(test_pair_probs_full[idx]),
        "tau": BEST_PAIR_OVERRIDE_TAU,
        "base_correct": base_correct,
        "final_correct": final_correct,
    })

pair_impact_df = pd.DataFrame(rows)

display(pair_impact_df)

if len(pair_impact_df) > 0:
    summary = pair_impact_df["effect"].value_counts().to_dict()
else:
    summary = {}

summary_df = pd.DataFrame([{
    "n_changed": int(test_changed.sum()),
    "n_corrected_errors": int(summary.get("corrected_error", 0)),
    "n_introduced_errors": int(summary.get("introduced_error", 0)),
    "n_changed_but_still_wrong": int(summary.get("changed_but_still_wrong", 0)),
    "n_changed_but_still_correct": int(summary.get("changed_but_still_correct", 0)),
}])

display(summary_df)

pair_override_search_df.to_csv(PAIR_OUTPUT_DIR / "pair_override_threshold_search.csv", index=False)
pair_specialist_comparison_df.to_csv(PAIR_OUTPUT_DIR / "pair_specialist_comparison.csv", index=False)
pair_impact_df.to_csv(PAIR_OUTPUT_DIR / "pair_specialist_changed_decisions.csv", index=False)
summary_df.to_csv(PAIR_OUTPUT_DIR / "pair_specialist_impact_summary.csv", index=False)

pd.DataFrame(classification_report(
    y_test_multiclass,
    final_pair_test_preds,
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0,
    output_dict=True,
)).transpose().to_csv(PAIR_OUTPUT_DIR / "classification_report_test.csv")

pd.DataFrame(
    confusion_matrix(y_test_multiclass, final_pair_test_preds, labels=list(range(NUM_CLASSES))),
    index=[f"true_{c}" for c in CLASS_NAMES],
    columns=[f"pred_{c}" for c in CLASS_NAMES],
).to_csv(PAIR_OUTPUT_DIR / "confusion_matrix_test.csv")

import shutil, os
from IPython.display import FileLink, display

zip_base = "/kaggle/working/stricture_lithiasis_specialist"
zip_path = Path(zip_base + ".zip")
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(zip_base, "zip", root_dir="/kaggle/working", base_dir="stricture_lithiasis_specialist")
os.chdir("/kaggle/working")
display(FileLink("stricture_lithiasis_specialist.zip"))


""


,n_changed,n_corrected_errors,n_introduced_errors,n_changed_but_still_wrong,n_changed_but_still_correct
0,0,0,0,0,0


/kaggle/working/stricture_lithiasis_specialist.zip